# Bioleft — Estandarización de las 4 campañas

EDA y mapeo de variables del dataset de observaciones agronómicas
(formato largo / EAV) de los ensayos participativos de tomate criollo.

## Setup

In [27]:
import pandas as pd
import re
import unicodedata

## 1. Auditoría de datos

Primer vistazo al dataset: cuántas filas, cuántas campañas y cuántas
variables únicas hay antes de empezar a estandarizar.

### 1.1 Leemos el dataset y hacemos una vista previa

In [29]:
df = pd.read_csv('Datos/observaciones_agronomicas.csv', encoding='utf-8-sig')
print(df.shape)
df.head(3)

(10313, 9)


,campaña,id_participante,id_sitio,variedad_id,variedad_original,etapa,variable,valor,nota
0,2024-2025,AA023T,AA023T,V172,172,Siembra y emergencia,Método de siembra,Siembra en almácigos,NaN
1,2024-2025,AA023T,AA023T,V172,172,Siembra y emergencia,Fecha de siembra,2024-09-02,NaN
2,2024-2025,AA023T,AA023T,V172,172,Siembra y emergencia,Cantidad de semillas sembradas,10,NaN


### 1.2 Observaciones por campaña

In [30]:
temporadas = df['campaña'].value_counts().sort_index()
print(f"Total de campañas: {df['campaña'].nunique()}")
temporadas

Total de campañas: 4


campaña
2022-2023    4156
2023-2024    2958
2024-2025    1495
2025-2026    1704
Name: count, dtype: int64

### 1.3 Variables únicas y su frecuencia (sin estandarizar todavía)

In [31]:
variables = df['variable'].value_counts().sort_index()
print(f"Total de variables únicas: {df['variable'].nunique()}")
variables

Total de variables únicas: 123


variable
#emergidas                                                                                                                116
#implantados                                                                                                               30
#plantines trasplantados                                                                                                   73
#por contenedor                                                                                                            62
#semillas sembradas                                                                                                       128
1ra fecha                                                                                                                  62
2da fecha                                                                                                                  37
3ra fecha                                                                                                    

Vemos muchas variables con nombres muy parecidos que pareciera que miden
lo mismo: serían variables duplicadas cuyo nombre fue variando de
campaña en campaña.

### 1.4 Detección automática de candidatos a duplicados

Normalizamos el nombre de la variable (minúsculas, sin tildes, sin
paréntesis, sin símbolos sueltos) y agrupamos por esa versión
normalizada para encontrar variables que probablemente sean la misma.

In [ ]:
def normalizar(texto):
    texto = texto.lower().strip()
    texto = re.sub(r'\(.*?\)', '', texto)  # sacamos lo que está entre paréntesis, ej: "(KG)"
    texto = re.sub(r'[#°ª]', '', texto)     # sacamos símbolos sueltos tipo "#"
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')  # sacamos tildes
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['variable_norm'] = df['variable'].apply(normalizar)

grupos = df.groupby('variable_norm')['variable'].unique()
candidatos_duplicados = grupos[grupos.apply(len) > 1]
candidatos_duplicados

variable_norm
cantidad de plantulas emergidas                                                                                                                                                            [Cantidad de plantulas emergidas, Cantidad de plántulas emergidas]
comentarios                                                                                                                                                                                                                        [Comentarios, comentarios]
conduccion                                                                                                                                                                                                                           [Conducción, Conduccion]
cuanto te gusta este tomate?                                                                         [¿Cuánto te gusta este tomate? (1-9), ¿Cuánto te gusta este tomate? (1 = me disgusta muchísimo, 5 = ni me gusta ni me disgu

- Con la detección automática, ya vemos que hay muchas variables con nombres muy parecidos y que pareciera que miden lo mismo: 
serían variables duplicadas cuyo nombre fue variando de campaña en campaña.

Vamos a empezar a mapear esas variables y después vamos a seguir investigando el resto de forma manual.

## 2. EDA — Mapeo y estandarización de variables

Objetivos de esta sección:
- Mapear y estandarizar variables duplicadas (miden lo mismo pero
  tienen distinto nombre según la campaña).
- Mapear y limpiar los valores que toman las variables (mayúsculas,
  tildes, espacios, etc. que representan lo mismo) 
- Tratar tipos de datos distintos o con unidades distintas dentro de
  un mismo atributo 
- Tratar datos faltantes

**Criterio de nombre estándar:** uso el nombre de la campaña más
reciente como estándar, y convierto los nombres de campañas
anteriores a ese estándar cuando difieren.

**Metodología:** para cada par de variables candidatas a ser la
misma, comparo los valores que toman (y en algunos casos la etapa
del cultivo en la que aparecen, porque puede haber nombres parecidos
que en realidad se refieren a etapas distintas) para confirmar que
realmente representan lo mismo antes de fusionarlas.

También se consulto al equipo de Bioleft para confirmar el mapeo de algunas variables, por ejemplo de "Fecha floracion" y "Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración)" y "Motivo" con "¿A qué crees que se debe la uniformidad observada?"

### 2.1 Comparaciones variable por variable

Cada sub-apartado revisa un par (o grupo) de variables candidatas a
fusionarse.

#### Cobertura vs. Cobertura de suelo

In [33]:
for var in ["Cobertura", "Cobertura de suelo"]:
    print(f"\n=== {var} ===")
    print(df[df["variable"] == var]["valor"].value_counts(dropna=False))


=== Cobertura ===
valor
vegetal              27
sin cobertura        27
cobertura vegetal     7
plástica              3
sin                   1
Name: count, dtype: int64

=== Cobertura de suelo ===
valor
cobertura vegetal    78
sin cobertura        40
0.17                  1
0.08                  1
Name: count, dtype: int64


#### Motivo vs. ¿A qué crees que se debe la uniformidad observada?

In [34]:
for var in ['Motivo', '¿A qué crees que se debe la uniformidad observada?']:
    print(f"\n=== {var} ===")
    print(df[df['variable']==var]['valor'].value_counts())


=== Motivo ===
valor
genética                                                                                                                                                       5
Tal vez el contenedor no fue el adecuado                                                                                                                       3
quizás no estaba bien tutorada                                                                                                                                 3
por su naturaleza y alimentación                                                                                                                               3
en su mayoria todas tienen un mismo crecimiento,en cuanto a tamaño y sanidad.                                                                                  2
Por la orientación algunas plantas reciben menos sol directo que otras.\t                                                                                      2
a las enferm

#### Implantados / trasplantados y Ubicación — comparación por etapa

Acá comparo por `etapa` en vez de por `valor`, justamente para
descartar que sean variables de etapas distintas.

In [36]:
for var in ['#implantados', 'Cantidad de plantas implantadas', 'Cantidad de plantines implantados',
            'Ubicación', 'Ubicación de la parcela', 'Ubicación del contenedor']:
    print(var, '->', df[df['variable']==var]['etapa'].unique())

#implantados -> ['Trasplante a maceta']
Cantidad de plantas implantadas -> ['Trasplante a suelo']
Cantidad de plantines implantados -> ['Trasplante a maceta']
Ubicación -> ['Trasplante a suelo']
Ubicación de la parcela -> ['Trasplante a suelo']
Ubicación del contenedor -> ['Trasplante a maceta']


#### Malezas / Plagas / Brote apical

In [38]:
for var in ['Malezas', 'Control de malezas', 'Plagas', 'Control de plagas', 'Plagas observadas',
            'Eliminación del brote apical', 'Brote apical']:
    print(f"\n=== {var} ===")
    print(df[df['variable']==var]['valor'].value_counts().head(8))


=== Malezas ===
valor
control mecánico         74
sin control              16
otro                     11
herbicidas sintéticos     2
Name: count, dtype: int64

=== Control de malezas ===
valor
control mecánico    108
sin control          13
Control Mecánico      4
control manual        2
manual                1
Name: count, dtype: int64

=== Plagas ===
valor
biopreparados              46
sin control                24
control mecánico           20
otro                        5
preparados biodinámicos     5
plaguicidas sintéticos      3
Name: count, dtype: int64

=== Control de plagas ===
valor
control mecánico           41
sin control                31
biopreparados              30
preparados biodinámicos    13
plaguicidas sintéticos      3
Manualmente                 3
otro                        3
Name: count, dtype: int64

=== Plagas observadas ===
valor
ninguna                                       8
Babosas                                       5
caracoles                        

#### Sustrato vs. 'Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.)'

In [39]:
for var in ['Sustrato', 'Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.)']:
    print(f"\n=== {var} ===")
    print(df[df['variable']==var]['valor'].value_counts().head(8))


=== Sustrato ===
valor
tierra, compost                    19
tierra, compost, perlita           16
tierra, perlita                     8
tierra, compost, turba, suelo       6
tierra, compost, perlita, suelo     5
tierra, compost, turba              5
compost                             4
compost, perlita, turba             4
Name: count, dtype: int64

=== Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.) ===
valor
Mezcla de tierra, compost y turba en proporciones iguales.    3
tierra, compost, perlita                                      3
tierra del patio, ceniza, y tierra negra comprada.            2
sustrato comercial grow y compost                             2
tierra                                                        1
Tierra + Compost                                              1
sustrato comercial grow y compost                             1
Name: count, dtype: int64


#### Tolerancia al sol

In [40]:
for var in ['Tolerancia sol', 'Tolerancia a la quemadura del sol (1 = nada tolerante, 5 = muy tolerante)']:
    print(f"\n=== {var} ===")
    print(df[df['variable']==var]['valor'].value_counts())


=== Tolerancia sol ===
valor
5      27
4      22
3      12
4.5    11
2.5     7
1       6
3.5     4
2       4
1.5     4
Name: count, dtype: int64

=== Tolerancia a la quemadura del sol (1 = nada tolerante, 5 = muy tolerante) ===
valor
4             41
3             23
5             20
4.5            7
2              6
04/05/2024     5
2.5            2
02/05/2024     2
03/05/2023     2
1              1
03/05/2024     1
01/05/2024     1
04/05/2023     1
Name: count, dtype: int64


#### Tolerancia a la sequía

In [41]:
for var in ['Tolerancia sequia', 'Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante)']:
    print(f"\n=== {var} ===")
    print(df[df['variable']==var]['valor'].value_counts())


=== Tolerancia sequia ===
valor
5      31
4      20
3      17
2.5    14
3.5     6
2       5
1       4
1.5     1
Name: count, dtype: int64

=== Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante) ===
valor
4             44
3             34
5             12
3.5            6
03/05/2024     4
02/05/2024     3
4.5            3
02/05/2023     2
2.5            2
1              2
04/05/2024     2
2              1
01/05/2024     1
03/05/2023     1
04/05/2023     1
Name: count, dtype: int64


### 2.2 Diccionario de mapeo de variables

Con base en las comparaciones anteriores (y en el resto de las
variables que ya eran claramente sinónimos por nombre), armé el
diccionario de mapeo. Está organizado por etapa del cultivo.

In [ ]:
mapeo_variables = {
    # --- Siembra y emergencia ---
    '#semillas sembradas': 'Cantidad de semillas sembradas',
    'Cantidad de plantulas emergidas': 'Cantidad de plántulas emergidas',
    '#emergidas': 'Cantidad de plántulas emergidas',

    'Fecha emergencia': 'Fecha de emergencia (cuando emerge el 70% de las plántulas)',
    'Fecha de emergencia': 'Fecha de emergencia (cuando emerge el 70% de las plántulas)',

    # --- Trasplante a maceta ---
    'Profundidad': 'Profundidad del contenedor (cm)',
    'Sustrato utilizado ': 'Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.)',  # Revisar cantidad de observaciones
    'Sustrato utilizado (marcá todo lo que hayas utilizado en tu contenedor)': 'Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.)',
    'Sustrato': 'Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.)',
    '#plantines trasplantados': 'Cantidad de plantines trasplantados',
    '#implantados': 'Cantidad de plantines implantados',

    # --- Trasplante a suelo ---
    'Ubicación': 'Ubicación de la parcela',
    'Superficie': 'Superficie de la parcela (m²)',
    'Densidad': 'Densidad (plantas/m²)',
    'Cobertura': 'Cobertura de suelo',

    # --- Labores ---
    'Conduccion': 'Conducción',
    'Fertilizacion': 'Fertilización',
    'Malezas': 'Control de malezas',
    'Plagas': 'Control de plagas',

    # --- Crecimiento y desarrollo ---
    'Fecha de primera floración': 'Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración)',
    'Fecha de primera floración ': 'Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración)',
    'Fecha floracion': 'Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración)',
    'Tolerancia sequia': 'Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante)',
    'Tolerancia sol': 'Tolerancia a la quemadura del sol (1 = nada tolerante, 5 = muy tolerante)',
    'Sanidad': 'Sanidad (1 = nada sana, 5 = muy sana)',
    'Uniformidad': 'Uniformidad (1 = nada uniforme, 5 = muy uniforme)',
    'Motivo': '¿A qué crees que se debe la uniformidad observada?',

    # --- Labores durante fructificación ---
    'Raleo': 'Raleo de frutos',
    'Brote apical': 'Eliminación del brote apical',

    # --- Cosecha y poscosecha ---
    '1ra fecha': 'Fecha de cosecha #1',  
    '2da fecha': 'Fecha de cosecha #2',
    '3ra fecha': 'Fecha de cosecha #3',
    '4ta fecha': 'Fecha de cosecha #4',
    '5ta fecha': 'Fecha de cosecha #5',
    '6ta fecha': 'Fecha de cosecha #6',
    'Color': 'Color del fruto al momento de cosechar',  # REVISAR: en la campaña 2022 tiene un número asignado, en las demás toma texto/string.
    'Poscosecha': 'Duración poscosecha (días que dura una muestra de 10 tomates en estado para ser consumido)', 
    'Peso de todos los frutos cosechados': 'Peso de todos los frutos cosechados (KG)',

    # --- Degustación ---
    'Olor maduro': 'Olor a tomate maduro',
    '¿Cuánto te gusta este tomate? (1-9)': '¿Cuánto te gusta este tomate? (1 = me disgusta muchísimo, 5 = ni me gusta ni me disgusta, 9 = me gusta muchísimo)',
    'Gusto': '¿Cuánto te gusta este tomate? (1 = me disgusta muchísimo, 5 = ni me gusta ni me disgusta, 9 = me gusta muchísimo)',
    'Olor verde': 'Olor a tomate verde',

    # --- Comentarios ---
    'comentarios': 'Comentarios generales',
    'Comentarios': 'Comentarios generales',
}

df['variable_limpia'] = df['variable'].replace(mapeo_variables)

Como tenemos en el dataset 2 variables con el mismo nombre pero provenientes de etapas distintas, vamos a hacer la distinción de ambas

In [43]:
mapeo_por_etapa = {
    'Trasplante a maceta': {
        'Fecha trasplante': 'Fecha de trasplante (maceta)',
        'Fecha de trasplante': 'Fecha de trasplante (maceta)',
    },
    'Trasplante a suelo': {
        'Fecha de trasplante': 'Fecha de trasplante (suelo)',
    },
}

def mapear(row):
    return mapeo_por_etapa.get(row['etapa'], {}).get(row['variable'], mapeo_variables.get(row['variable'], row['variable']))

df['variable_limpia'] = df.apply(mapear, axis=1)

In [44]:
print(f"Antes: {df['variable'].nunique()} — Después: {df['variable_limpia'].nunique()}")
df.head(2)

Antes: 123 — Después: 79


,campaña,id_participante,id_sitio,variedad_id,variedad_original,etapa,variable,valor,nota,variable_norm,variable_limpia
0,2024-2025,AA023T,AA023T,V172,172,Siembra y emergencia,Método de siembra,Siembra en almácigos,NaN,metodo de siembra,Método de siembra
1,2024-2025,AA023T,AA023T,V172,172,Siembra y emergencia,Fecha de siembra,2024-09-02,NaN,fecha de siembra,Fecha de siembra


## 3. Resultado del mapeo

### 3.1 Variables únicas después de estandarizar

In [45]:
variables_limpias = df['variable_limpia'].value_counts().sort_index()
print(f"Total de variables únicas: {df['variable_limpia'].nunique()}")
for var, cant in variables_limpias.items():
    print(f"{var}: {cant}")

Total de variables únicas: 79
#por contenedor: 62
Aguachento: 49
Cantidad de frutos cosechados: 67
Cantidad de plantas implantadas: 53
Cantidad de plantas trasplantadas a la parcela: 117
Cantidad de plantines implantados: 54
Cantidad de plantines trasplantados: 157
Cantidad de plántulas emergidas: 302
Cantidad de semillas sembradas: 342
Cobertura de suelo: 185
Color del fruto al momento de cosechar: 197
Comentarios generales: 778
Conducción: 202
Control de malezas: 231
Control de plagas: 227
Cultivo antecesor: 56
Cultivo asociado: 203
Densidad (plantas/m²): 117
Desbrote: 241
Distancia plantas: 59
Distancia surcos: 51
Dulce: 123
Duración poscosecha (días que dura una muestra de 10 tomates en estado para ser consumido): 100
Eliminación del brote apical: 219
Fecha de cosecha #1: 148
Fecha de cosecha #10: 14
Fecha de cosecha #2: 106
Fecha de cosecha #3: 73
Fecha de cosecha #4: 48
Fecha de cosecha #5: 41
Fecha de cosecha #6: 34
Fecha de cosecha #7: 23
Fecha de cosecha #8: 20
Fecha de cosech

### 3.2 Estado por etapa del cultivo

Variables ya resueltas completamente (mapeadas y confirmadas) vs.
variables que todavía hay que resolver, agrupadas por etapa.

### Resueltas completamente

- Comentarios generales: 778

##### Siembra y emergencia — 5/5
- Método de siembra: 139
- Fecha de siembra: 344
- Cantidad de semillas sembradas: 342
- Cantidad de plántulas emergidas: 302
- Fecha de emergencia (cuando emerge el 70% de las plántulas): 278

##### Trasplante a maceta — 9/9
- Ubicación del contenedor: 166
- Profundidad del contenedor (cm): 132
- Sustrato utilizado (...): 253
- Fecha de trasplante (maceta): 146
- Cantidad de plantines trasplantados: 157
- Cantidad de plantines implantados: 54
- Cultivo asociado: 62
- Volumen: 57
- #por contenedor: 62 (probablemente plantines trasplantados por contenedor, solo en 2022-2023)

##### Trasplante a suelo — 12/12
- Ubicación de la parcela: 189
- Superficie de la parcela (m²): 141
- Fecha de trasplante (suelo): 118
- Cantidad de plantas trasplantadas a la parcela: 117
- Densidad (plantas/m²): 117
- Cobertura de suelo: 185
- Cultivo asociado: 141
- Cantidad de plantas implantadas: 53
- Surcos: 53 (solo en 2022-2023 — cantidad de surcos)
- Distancia surcos: 51 (solo en 2022-2023 — distancia entre surcos)
- Distancia plantas: 59 (solo en 2022-2023 — distancia entre plantas de un mismo surco)
- Cultivo antecesor: 56

##### Labores — 7/7
- Desbrote: 241
- Tutorado: 247
- Conducción: 202
- Fertilización: 243
- Control de malezas: 231
- Riego: 256
- Control de plagas: 227

##### Crecimiento y desarrollo — 8/8
- Fecha de primera floración (día en la que al menos la mitad de las plantas están en su primera floración): 172
- Tolerancia al estrés hídrico (1 = nada tolerante, 5 = muy tolerante): 216
- Tolerancia a la quemadura del sol (1 = nada tolerante, 5 = muy tolerante): 209
- Sanidad (1 = nada sana, 5 = muy sana): 217
- Muerte por virus rugoso del tomate: 106
- Uniformidad (1 = nada uniforme, 5 = muy uniforme): 184
- ¿A qué crees que se debe la uniformidad observada?: 70
- Plagas observadas: 81

##### Labores durante fructificación — 2/2
- Raleo de frutos: 182
- Eliminación del brote apical: 219

##### Cosecha y poscosecha — 14/20
- Fecha de cosecha #1 a #10: 148, 106, 73, 48, 41, 34, 23, 20, 17, 14
- Color del fruto al momento de cosechar: 197
- Duración poscosecha (...): 100
- Cantidad de frutos cosechados: 67
- Peso de todos los frutos cosechados (KG): 57

##### Degustación — 16/16
- ¿Cuánto te gusta este tomate? (...): 164
- Dulce: 123 / Ácido: 87 / Jugoso: 154 / Persistente: 80
- Olor a tomate maduro: 121 / Olor a tomate verde: 32
- Mucha pulpa: 99 / Poca pulpa: 68 / Pulpa dura: 43 / Pulpa blanda: 102
- Mucha semilla: 108 / Poca semilla: 59
- Aguachento: 49 / Pastoso: 34 / Piel dura: 104

#### Pendientes de resolver
- Rendimiento 1 a 6: 38, 28, 12, 3, 3, 2 (todas de la campaña 2022-2023). Peso de todos los frutos en kilos de esa fecha. 

### 3.3 Listado ordenado por etapa

Mismo resultado que 3.1, pero agrupado por etapa del cultivo

In [46]:
for etapa in df['etapa'].unique():
    for var in df[df['etapa']==etapa]['variable_limpia'].unique():
        print(etapa, '->', var)

Siembra y emergencia -> Método de siembra
Siembra y emergencia -> Fecha de siembra
Siembra y emergencia -> Cantidad de semillas sembradas
Siembra y emergencia -> Cantidad de plántulas emergidas
Siembra y emergencia -> Fecha de emergencia (cuando emerge el 70% de las plántulas)
Siembra y emergencia -> Comentarios generales
Trasplante a maceta -> Ubicación del contenedor
Trasplante a maceta -> Profundidad del contenedor (cm)
Trasplante a maceta -> Sustrato utilizado (Indicar todos los utilizados. Ejemplos: Tierra, compost, vermiculita, arena, turba, suelo, otros.)
Trasplante a maceta -> Fecha de trasplante (maceta)
Trasplante a maceta -> Cantidad de plantines trasplantados
Trasplante a maceta -> Comentarios generales
Trasplante a maceta -> Cantidad de plantines implantados
Trasplante a maceta -> Cultivo asociado
Trasplante a maceta -> Volumen
Trasplante a maceta -> #por contenedor
Labores -> Desbrote
Labores -> Tutorado
Labores -> Conducción
Labores -> Fertilización
Labores -> Control de